# Lab 7: Retrieval, Tools, and Agent Interfaces

Earlier labs focused on how models reason and how we evaluate their outputs. In this lab, we study a different question: **how can we extend a language model's capabilities by connecting it to external knowledge and tools?**

You will progressively build a system around a language model by adding components:

1. **Direct answering**: the model answers using parametric knowledge
2. **Retrieval-Augmented Generation (RAG)**: retrieve documents from a local corpus
3. **Tool calling**: connect the model to external tools such as a calculator
4. **Internet search**: add an external search tool using DuckDuckGo
5. **Agent routing**: decide when to answer directly, retrieve, or call a tool
6. **Trace logging**: log and visualize agent behavior
7. **Tool description experiments**: modify tool descriptions and observe routing behavior

The key insight: **modern AI systems are not just models. They are systems built around models that interact with knowledge and environments.**

**This is the solved version with all implementations filled in.**

---

---

## Section 0: Setup

We will use [OpenRouter](https://openrouter.ai) to access language models. OpenRouter provides a unified API compatible with the OpenAI client library.

### Install Dependencies

In [1]:
!pip install openai sentence-transformers faiss-cpu pydantic duckduckgo-search

### Imports and Configuration

In [2]:
import os
import re
import json
import time
import numpy as np
from pathlib import Path
from openai import OpenAI

# --- OpenRouter Configuration ---
API_KEY = os.getenv("OPENROUTER_API_KEY", "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")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
)

MODEL = "google/gemini-2.0-flash-001"

print(f"Using model: {MODEL}")
print(f"API key configured: {'Yes' if API_KEY != 'your-key-here' else 'No -- set OPENROUTER_API_KEY'}")

Using model: google/gemini-2.0-flash-001
API key configured: Yes


### Helper: `call_model()`

This helper wraps the OpenAI chat completion call. It handles messages, optional tool definitions, and returns the response object.

**PROVIDED** -- do not modify.

In [3]:
def call_model(messages, tools=None, model=MODEL):
    """Call the language model via OpenRouter.
    
    Args:
        messages: list of message dicts ({"role": ..., "content": ...})
        tools: optional list of tool schemas for function calling
        model: model identifier
    
    Returns:
        The API response object.
    """
    kwargs = {
        "model": model,
        "messages": messages,
    }
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    
    response = client.chat.completions.create(**kwargs)
    return response

### API Connection Test

In [4]:
# Quick test to verify the API connection works
test_response = call_model([{"role": "user", "content": "Say 'hello' and nothing else."}])
print("API test:", test_response.choices[0].message.content)
print("Connection successful!")

API test: Hello.

Connection successful!


---

## Section 1: Direct Question Answering

Our simplest system: send the user's query directly to the model and return its response.

```
user query  -->  model  -->  answer
```

This baseline uses only the model's **parametric knowledge** -- whatever it learned during training.

In [5]:
def ask_direct(query):
    """Ask the model a question directly (no retrieval, no tools).
    
    Args:
        query: the user's question
    
    Returns:
        dict with keys: 'response' (str), 'total_tokens' (int)
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer the user's question concisely."},
        {"role": "user", "content": query}
    ]
    response = call_model(messages)
    content = response.choices[0].message.content
    total_tokens = response.usage.total_tokens if response.usage else 0
    return {"response": content, "total_tokens": total_tokens}

In [6]:
# Test direct QA on several questions
test_queries = [
    "What is chain-of-thought prompting?",
    "What is retrieval augmented generation?",
    "What is sqrt(98765)?",
    "When was the Toolformer paper published?",
]

print("=" * 70)
print("DIRECT QA RESULTS")
print("=" * 70)
direct_results = {}
for q in test_queries:
    result = ask_direct(q)
    direct_results[q] = result
    print(f"\nQ: {q}")
    print(f"A: {result['response'][:200]}")
    print(f"Tokens: {result['total_tokens']}")
    print("-" * 70)

DIRECT QA RESULTS



Q: What is chain-of-thought prompting?
A: Chain-of-thought prompting is a technique to improve the reasoning abilities of large language models by prompting them to explain their reasoning process step-by-step before answering a question.

Tokens: 60
----------------------------------------------------------------------



Q: What is retrieval augmented generation?
A: Retrieval-Augmented Generation (RAG) is an AI framework that enhances the accuracy and reliability of generative AI models by grounding them in external knowledge sources. It involves retrieving relev
Tokens: 78
----------------------------------------------------------------------



Q: What is sqrt(98765)?
A: sqrt(98765) ≈ 314.268927

Tokens: 45
----------------------------------------------------------------------



Q: When was the Toolformer paper published?
A: The Toolformer paper was published in **February 2023**.

Tokens: 38
----------------------------------------------------------------------


**Observation:** The model can answer general knowledge questions, but may struggle with:
- Precise calculations (sqrt of large numbers)
- Specific factual details (exact publication dates)
- Information from specific documents

We will address each of these limitations in the following sections.

---

## Section 2: Retrieval-Augmented Generation (RAG)

RAG augments the model with external knowledge by retrieving relevant documents before generating an answer.

```
documents  -->  chunking  -->  embeddings  -->  vector index
                                                    |
user query  -->  embed query  -->  search index  -->  retrieved chunks  -->  model  -->  answer
```

### Load Documents

**PROVIDED** -- loads all `.txt` files from the `docs/` directory.

In [7]:
def load_documents(doc_dir="docs"):
    """Load all .txt files from the document directory.
    
    Returns:
        list of dicts with keys: 'filename', 'text'
    """
    documents = []
    doc_path = Path(doc_dir)
    for filepath in sorted(doc_path.glob("*.txt")):
        text = filepath.read_text(encoding="utf-8")
        documents.append({"filename": filepath.name, "text": text})
    print(f"Loaded {len(documents)} documents from {doc_dir}/")
    for doc in documents:
        print(f"  - {doc['filename']} ({len(doc['text'])} chars)")
    return documents

documents = load_documents()

Loaded 6 documents from docs/
  - chain_of_thought.txt (1624 chars)
  - function_calling.txt (2207 chars)
  - language_model_agents.txt (2302 chars)
  - prompt_engineering.txt (1963 chars)
  - retrieval_augmented_generation.txt (1820 chars)
  - toolformer.txt (1948 chars)


### Chunk Documents

Split each document into overlapping chunks of fixed character length. This ensures that each chunk fits within an embedding model's context and that information at chunk boundaries is not lost.

In [8]:
def chunk_documents(documents, chunk_size=500, overlap=50):
    """Split documents into overlapping chunks.
    
    Args:
        documents: list of dicts with 'filename' and 'text'
        chunk_size: number of characters per chunk
        overlap: number of overlapping characters between consecutive chunks
    
    Returns:
        list of dicts with keys: 'text' (chunk text), 'source' (filename)
    """
    chunks = []
    for doc in documents:
        text = doc["text"]
        filename = doc["filename"]
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]
            chunks.append({"text": chunk_text, "source": filename})
            start += chunk_size - overlap
    return chunks

chunks = chunk_documents(documents)
print(f"Created {len(chunks)} chunks")
print(f"Example chunk (first 100 chars): {chunks[0]['text'][:100]}...")
print(f"Source: {chunks[0]['source']}")

Created 30 chunks
Example chunk (first 100 chars): Chain-of-Thought Prompting

Chain-of-thought (CoT) prompting is a technique introduced by Wei et al....
Source: chain_of_thought.txt


### Build Vector Index

Encode all chunks using a sentence embedding model and build a FAISS index for fast nearest-neighbor search.

In [9]:
import faiss
from sentence_transformers import SentenceTransformer

def build_index(chunks):
    """Build a FAISS vector index from document chunks.
    
    Args:
        chunks: list of chunk dicts with 'text' and 'source'
    
    Returns:
        tuple of (faiss_index, chunks, embedding_model)
    """
    model = SentenceTransformer("all-MiniLM-L6-v2")
    texts = [chunk["text"] for chunk in chunks]
    embeddings = model.encode(texts, show_progress_bar=True)
    embeddings = np.array(embeddings).astype("float32")
    
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    
    print(f"Built FAISS index with {index.ntotal} vectors of dimension {dimension}")
    return index, chunks, model

faiss_index, indexed_chunks, embed_model = build_index(chunks)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Built FAISS index with 30 vectors of dimension 384


### Retrieve Relevant Chunks

In [10]:
def retrieve(query, index, chunks, model, k=3):
    """Retrieve the top-k most relevant chunks for a query.
    
    Args:
        query: the search query string
        index: the FAISS index
        chunks: the list of chunk dicts
        model: the SentenceTransformer model
        k: number of results to return
    
    Returns:
        list of chunk dicts (top-k most relevant)
    """
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, k)
    results = [chunks[i] for i in indices[0]]
    return results

# Test retrieval
test_results = retrieve("chain-of-thought prompting", faiss_index, indexed_chunks, embed_model)
for i, chunk in enumerate(test_results):
    print(f"\nResult {i+1} (source: {chunk['source']}):")
    print(f"  {chunk['text'][:150]}...")


Result 1 (source: chain_of_thought.txt):
   phrase "Let's think step by step" to the prompt, which has been shown to elicit reasoning without any examples.

Chain-of-thought prompting works bes...

Result 2 (source: chain_of_thought.txt):
  Chain-of-Thought Prompting

Chain-of-thought (CoT) prompting is a technique introduced by Wei et al. (2022) that improves the reasoning ability of lar...

Result 3 (source: prompt_engineering.txt):
  System prompts are particularly important in chat-based applications where they set the context for an entire conversation.

More advanced techniques ...


### Ask with Retrieval

Combine retrieval with generation: retrieve relevant chunks, include them as context in the prompt, and ask the model to answer based on the provided context.

In [11]:
def ask_with_retrieval(query, index, chunks, model):
    """Answer a question using retrieval-augmented generation.
    
    Args:
        query: the user's question
        index: the FAISS index
        chunks: the list of chunk dicts
        model: the SentenceTransformer embedding model
    
    Returns:
        dict with keys: 'response', 'retrieved_chunks', 'total_tokens'
    """
    retrieved = retrieve(query, index, chunks, model, k=3)
    
    context = "\n\n---\n\n".join(
        f"[Source: {chunk['source']}]\n{chunk['text']}" for chunk in retrieved
    )
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer the user's question based on the provided context. If the context does not contain the answer, say so."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]
    
    response = call_model(messages)
    content = response.choices[0].message.content
    total_tokens = response.usage.total_tokens if response.usage else 0
    
    return {
        "response": content,
        "retrieved_chunks": retrieved,
        "total_tokens": total_tokens
    }

### Compare RAG vs Direct

In [12]:
print("=" * 70)
print("RAG vs DIRECT COMPARISON")
print("=" * 70)

for q in test_queries:
    rag_result = ask_with_retrieval(q, faiss_index, indexed_chunks, embed_model)
    direct_result = direct_results[q]
    
    print(f"\nQ: {q}")
    print(f"\n  Direct: {direct_result['response'][:200]}")
    print(f"  Tokens: {direct_result['total_tokens']}")
    print(f"\n  RAG:    {rag_result['response'][:200]}")
    print(f"  Tokens: {rag_result['total_tokens']}")
    print(f"  Sources: {[c['source'] for c in rag_result['retrieved_chunks']]}")
    print("-" * 70)

RAG vs DIRECT COMPARISON



Q: What is chain-of-thought prompting?

  Direct: Chain-of-thought prompting is a technique to improve the reasoning abilities of large language models by prompting them to explain their reasoning process step-by-step before answering a question.

  Tokens: 60

  RAG:    Chain-of-thought (CoT) prompting is a technique that improves the reasoning ability of large language models by providing step-by-step reasoning examples in the prompt. It encourages the model to brea
  Tokens: 431
  Sources: ['chain_of_thought.txt', 'chain_of_thought.txt', 'prompt_engineering.txt']
----------------------------------------------------------------------



Q: What is retrieval augmented generation?

  Direct: Retrieval-Augmented Generation (RAG) is an AI framework that enhances the accuracy and reliability of generative AI models by grounding them in external knowledge sources. It involves retrieving relev
  Tokens: 78

  RAG:    Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation to improve the factual accuracy and knowledge coverage of language models. In a RAG system,
  Tokens: 460
  Sources: ['retrieval_augmented_generation.txt', 'retrieval_augmented_generation.txt', 'retrieval_augmented_generation.txt']
----------------------------------------------------------------------



Q: What is sqrt(98765)?

  Direct: sqrt(98765) ≈ 314.268927

  Tokens: 45

  RAG:    This document discusses function calling in large language models and does not contain information about how to calculate the square root of 98765.

  Tokens: 387
  Sources: ['function_calling.txt', 'function_calling.txt', 'function_calling.txt']
----------------------------------------------------------------------



Q: When was the Toolformer paper published?

  Direct: The Toolformer paper was published in **February 2023**.

  Tokens: 38

  RAG:    The Toolformer paper was published in February 2023.

  Tokens: 378
  Sources: ['toolformer.txt', 'toolformer.txt', 'function_calling.txt']
----------------------------------------------------------------------


**Observation:** RAG improves answers for questions that relate to information in our document corpus. However, it does not help with:
- Math calculations (retrieval cannot compute)
- Real-time information (our corpus is static)

---

## Section 3: Tools

Tools extend the model's capabilities beyond text generation. We implement two tools:

1. **Calculator**: evaluate mathematical expressions
2. **Internet search**: look up current information via DuckDuckGo

### Calculator Tool

In [13]:
def calculator(expression):
    """Evaluate a mathematical expression safely.
    
    Only allows digits, operators (+, -, *, /, **, %), parentheses,
    spaces, and decimal points.
    
    Args:
        expression: a string containing the math expression
    
    Returns:
        dict with keys: 'result' (numeric value), 'expression' (input string)
    """
    try:
        # Only allow safe characters: digits, operators, parens, spaces, dots
        if not re.match(r'^[\d+\-*/().\s%]+$', expression):
            return {"error": f"Invalid expression: {expression}", "expression": expression}
        
        result = eval(expression)
        return {"result": result, "expression": expression}
    except Exception as e:
        return {"error": str(e), "expression": expression}

# Test calculator
print(calculator("15 * 37"))
print(calculator("98765 ** 0.5"))
print(calculator("(3 + 4) * (5 - 2)"))
print(calculator("144 ** 0.5"))

{'result': 555, 'expression': '15 * 37'}
{'result': 314.2689930616764, 'expression': '98765 ** 0.5'}
{'result': 21, 'expression': '(3 + 4) * (5 - 2)'}
{'result': 12.0, 'expression': '144 ** 0.5'}


### Internet Search Tool

In [14]:
from duckduckgo_search import DDGS

def internet_search(query, k=3, retries=2):
    """Search the internet using DuckDuckGo.
    
    Args:
        query: the search query string
        k: number of results to return
        retries: number of retry attempts if results are empty
    
    Returns:
        list of dicts with keys: 'title', 'snippet', 'url'
    """
    for attempt in range(retries + 1):
        try:
            results = []
            with DDGS() as ddgs:
                for r in ddgs.text(query, max_results=k):
                    results.append({
                        "title": r["title"],
                        "snippet": r["body"],
                        "url": r["href"]
                    })
            if results:
                return results
            time.sleep(1)
        except Exception as e:
            if attempt == retries:
                return [{"error": str(e)}]
            time.sleep(1)
    return []

# Test internet search
search_results = internet_search("Toolformer paper publication date")
for r in search_results:
    print(f"Title: {r.get('title', 'N/A')}")
    print(f"Snippet: {r.get('snippet', 'N/A')[:150]}")
    print(f"URL: {r.get('url', 'N/A')}")
    print()

---

## Section 4: Function Calling

Function calling lets the model output structured tool invocations. The system:

1. Sends the query and tool schemas to the model
2. The model decides whether to call a tool and generates structured arguments
3. We execute the tool call locally
4. We send the tool result back to the model
5. The model generates a final answer incorporating the tool result

```
user query + tool schemas  -->  model  -->  tool_call(name, args)
                                                 |
                                           execute tool
                                                 |
                                tool result  -->  model  -->  final answer
```

### Define Tool Schemas

In [15]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression. Use for arithmetic, algebra, square roots, etc.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to evaluate, e.g. '15 * 37' or '144 ** 0.5'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "internet_search",
            "description": "Search the internet for current information, recent events, or facts not in the local knowledge base.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print(f"Defined {len(TOOLS)} tool schemas:")
for tool in TOOLS:
    print(f"  - {tool['function']['name']}: {tool['function']['description']}")

Defined 2 tool schemas:
  - calculator: Evaluate a mathematical expression. Use for arithmetic, algebra, square roots, etc.
  - internet_search: Search the internet for current information, recent events, or facts not in the local knowledge base.


### Tool Call Dispatcher

**PROVIDED** -- routes a tool call to the correct function.

In [16]:
def execute_tool_call(tool_name, arguments):
    """Execute a tool call by dispatching to the correct function.
    
    Args:
        tool_name: name of the tool to call
        arguments: dict of arguments to pass to the tool
    
    Returns:
        The tool's return value (dict or list)
    """
    if tool_name == "calculator":
        return calculator(arguments["expression"])
    elif tool_name == "internet_search":
        return internet_search(arguments.get("query", ""))
    else:
        return {"error": f"Unknown tool: {tool_name}"}

### Ask with Tools

Send the query with tool schemas. If the model returns a tool call, execute it and send the result back.

In [17]:
def ask_with_tools(query, tools=TOOLS):
    """Answer a question using function calling.
    
    Args:
        query: the user's question
        tools: list of tool schemas
    
    Returns:
        dict with keys: 'response', 'tool_calls' (list), 'total_tokens'
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the provided tools when appropriate to answer the user's question accurately."},
        {"role": "user", "content": query}
    ]
    
    response = call_model(messages, tools=tools)
    message = response.choices[0].message
    total_tokens = response.usage.total_tokens if response.usage else 0
    
    tool_calls_log = []
    
    # Check if the model wants to call tools
    if message.tool_calls:
        # Add the assistant message with tool calls to the conversation
        messages.append(message)
        
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            # Execute the tool
            tool_result = execute_tool_call(tool_name, arguments)
            tool_calls_log.append({
                "tool": tool_name,
                "arguments": arguments,
                "result": tool_result
            })
            
            # Add tool result to messages
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result)
            })
        
        # Call model again with tool results
        response2 = call_model(messages)
        content = response2.choices[0].message.content
        total_tokens += response2.usage.total_tokens if response2.usage else 0
    else:
        content = message.content
    
    return {
        "response": content,
        "tool_calls": tool_calls_log,
        "total_tokens": total_tokens
    }

In [18]:
# Test function calling with math and search queries
tool_test_queries = [
    "What is the square root of 98765?",
    "What is 1234 * 5678?",
    "When was the Toolformer paper published?",
]

print("=" * 70)
print("FUNCTION CALLING RESULTS")
print("=" * 70)
for q in tool_test_queries:
    result = ask_with_tools(q)
    print(f"\nQ: {q}")
    print(f"A: {result['response'][:200]}")
    if result['tool_calls']:
        for tc in result['tool_calls']:
            print(f"  Tool: {tc['tool']}({tc['arguments']}) -> {tc['result']}")
    print(f"Tokens: {result['total_tokens']}")
    print("-" * 70)

FUNCTION CALLING RESULTS



Q: What is the square root of 98765?
A: The square root of 98765 is approximately 314.269.

  Tool: calculator({'expression': '98765 ** 0.5'}) -> {'result': 314.2689930616764, 'expression': '98765 ** 0.5'}
Tokens: 223
----------------------------------------------------------------------



Q: What is 1234 * 5678?
A: 1234 * 5678 = 7006652

  Tool: calculator({'expression': '1234 * 5678'}) -> {'result': 7006652, 'expression': '1234 * 5678'}
Tokens: 222
----------------------------------------------------------------------


/var/folders/2x/lt99d9v97mgfkbc813jw5cdw0000gn/T/ipykernel_8829/2586427243.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Q: When was the Toolformer paper published?
A: ```tool_code
print(default_api.internet_search(query = "when was Toolformer paper published?"))
```
  Tool: internet_search({'query': 'Toolformer paper publication date'}) -> []
Tokens: 201
----------------------------------------------------------------------


---

## Section 5: Agent Router

Now we build a simple agent that **routes** each query to the best handler:

- **direct**: general knowledge questions
- **retrieval**: questions about topics in our document corpus
- **calculator**: math problems
- **internet_search**: questions requiring current/real-time information

The router itself is an LLM call: we ask the model to classify the query.

In [19]:
def route_query(query):
    """Classify a query into one of four routes.
    
    Args:
        query: the user's question
    
    Returns:
        str: one of 'direct', 'retrieval', 'calculator', 'internet_search'
    """
    system_prompt = """You are a query router. Classify the user's query into exactly one category.

Categories:
- "direct": General knowledge questions the model can answer from training data.
- "retrieval": Questions about specific AI/ML topics like chain-of-thought prompting, RAG, prompt engineering, function calling, toolformer, or language model agents. These topics are covered in our document corpus.
- "calculator": Questions that require mathematical computation (arithmetic, algebra, square roots, etc.).
- "internet_search": Questions about current events, recent news, real-time information, or specific facts that require up-to-date information.

Respond with ONLY the category name, nothing else."""
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]
    
    response = call_model(messages)
    route = response.choices[0].message.content.strip().lower().strip('"')
    
    # Validate route
    valid_routes = ["direct", "retrieval", "calculator", "internet_search"]
    if route not in valid_routes:
        route = "direct"  # Default fallback
    
    return route

# Test routing
routing_tests = [
    "What is chain-of-thought prompting?",
    "What is 42 * 58?",
    "Who won the most recent Super Bowl?",
    "What is the capital of France?",
]

for q in routing_tests:
    route = route_query(q)
    print(f"  {route:20s} <- {q}")

  retrieval            <- What is chain-of-thought prompting?


  calculator           <- What is 42 * 58?


  internet_search      <- Who won the most recent Super Bowl?


  direct               <- What is the capital of France?


In [20]:
def agent_answer(query, index, chunks, embed_model, tools=TOOLS):
    """Route a query and dispatch to the appropriate handler.
    
    Args:
        query: the user's question
        index: FAISS index for retrieval
        chunks: indexed chunks for retrieval
        embed_model: SentenceTransformer model for retrieval
        tools: tool schemas for function calling
    
    Returns:
        dict with keys: 'query', 'route', 'answer', 'total_tokens'
    """
    route = route_query(query)
    
    if route == "direct":
        result = ask_direct(query)
        answer = result["response"]
        total_tokens = result["total_tokens"]
    elif route == "retrieval":
        result = ask_with_retrieval(query, index, chunks, embed_model)
        answer = result["response"]
        total_tokens = result["total_tokens"]
    elif route == "calculator":
        calc_tools = [t for t in tools if t["function"]["name"] == "calculator"]
        result = ask_with_tools(query, tools=calc_tools)
        answer = result["response"]
        total_tokens = result["total_tokens"]
    elif route == "internet_search":
        search_tools = [t for t in tools if t["function"]["name"] == "internet_search"]
        result = ask_with_tools(query, tools=search_tools)
        answer = result["response"]
        total_tokens = result["total_tokens"]
    else:
        result = ask_direct(query)
        answer = result["response"]
        total_tokens = result["total_tokens"]
    
    return {
        "query": query,
        "route": route,
        "answer": answer,
        "total_tokens": total_tokens
    }

In [21]:
# Test the agent on diverse queries
agent_test_queries = [
    "What is chain-of-thought prompting?",
    "What is sqrt(98765)?",
    "Who won the most recent Super Bowl?",
    "What is the capital of France?",
    "What is retrieval augmented generation?",
    "What is 2**10?",
]

print("=" * 70)
print("AGENT ROUTER RESULTS")
print("=" * 70)
for q in agent_test_queries:
    result = agent_answer(q, faiss_index, indexed_chunks, embed_model)
    print(f"\nQ: {result['query']}")
    print(f"Route: {result['route']}")
    print(f"A: {result['answer'][:200]}")
    print(f"Tokens: {result['total_tokens']}")
    print("-" * 70)

AGENT ROUTER RESULTS



Q: What is chain-of-thought prompting?
Route: retrieval
A: Chain-of-thought (CoT) prompting is a technique that improves the reasoning ability of large language models by providing step-by-step reasoning examples in the prompt. It encourages the model to brea
Tokens: 460
----------------------------------------------------------------------



Q: What is sqrt(98765)?
Route: calculator
A: The square root of 98765 is approximately 314.27.

Tokens: 187
----------------------------------------------------------------------


/var/folders/2x/lt99d9v97mgfkbc813jw5cdw0000gn/T/ipykernel_8829/2586427243.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Q: Who won the most recent Super Bowl?
Route: internet_search
A: I am sorry, I do not have the information for the winner of the most recent Super Bowl. To provide you with the most accurate answer, I need to know which Super Bowl you are asking about. Please speci
Tokens: 184
----------------------------------------------------------------------



Q: What is the capital of France?
Route: direct
A: The capital of France is Paris.

Tokens: 29
----------------------------------------------------------------------



Q: What is retrieval augmented generation?
Route: retrieval
A: Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation to improve the factual accuracy and knowledge coverage of language models. In a RAG system,
Tokens: 460
----------------------------------------------------------------------



Q: What is 2**10?
Route: calculator
A: 2**10 is 1024.

Tokens: 158
----------------------------------------------------------------------


---

## Section 6: Trace Logging

To understand and debug agent behavior, we log every step the agent takes. A **trace** records the full sequence of decisions, tool calls, and responses.

### Trace Class

In [22]:
class Trace:
    """Records the steps of an agent's execution."""

    def __init__(self, query):
        self.query = query
        self.steps = []
        self.route = None
        self.final_answer = None
        self.start_time = time.time()

    def add_step(self, step_type, input_data, output_data, metadata=None):
        """Add a step to the trace.

        Args:
            step_type: str describing the step (e.g., 'route', 'retrieval', 'tool_call', 'llm_call')
            input_data: the input to this step
            output_data: the output of this step
            metadata: optional dict of extra info
        """
        step = {
            "step_type": step_type,
            "input": input_data,
            "output": output_data,
            "timestamp": time.time() - self.start_time,
        }
        if metadata:
            step["metadata"] = metadata
        self.steps.append(step)

    def to_dict(self):
        """Convert the trace to a JSON-serializable dict."""
        return {
            "query": self.query,
            "route": self.route,
            "final_answer": self.final_answer,
            "steps": self.steps,
            "total_time": time.time() - self.start_time,
        }

    def save(self, filename):
        """Save the trace to a JSON file in the traces/ directory."""
        traces_dir = Path("traces")
        traces_dir.mkdir(exist_ok=True)
        filepath = traces_dir / filename
        with open(filepath, "w") as f:
            json.dump(self.to_dict(), f, indent=2, default=str)
        print(f"Trace saved to {filepath}")

### Traced Agent

Same as `agent_answer` but creates and populates a `Trace` at each step.

In [23]:
def agent_answer_traced(query, index, chunks, embed_model, tools=TOOLS):
    """Route and answer a query, logging each step in a Trace.

    Args:
        query: the user's question
        index: FAISS index
        chunks: indexed chunks
        embed_model: SentenceTransformer model
        tools: tool schemas

    Returns:
        tuple of (result_dict, trace)
    """
    trace = Trace(query)

    # Step 1: Route
    route = route_query(query)
    trace.route = route
    trace.add_step("route", {"query": query}, {"route": route})

    # Step 2: Execute based on route
    if route == "direct":
        result = ask_direct(query)
        trace.add_step("llm_call", {"query": query, "method": "direct"},
                       {"response": result["response"][:200]},
                       metadata={"tokens": result["total_tokens"]})
        answer = result["response"]
        total_tokens = result["total_tokens"]

    elif route == "retrieval":
        retrieved = retrieve(query, index, chunks, embed_model, k=3)
        trace.add_step("retrieval", {"query": query},
                       {"num_chunks": len(retrieved), "sources": [c["source"] for c in retrieved]})
        result = ask_with_retrieval(query, index, chunks, embed_model)
        trace.add_step("llm_call", {"query": query, "method": "rag"},
                       {"response": result["response"][:200]},
                       metadata={"tokens": result["total_tokens"]})
        answer = result["response"]
        total_tokens = result["total_tokens"]

    elif route == "calculator":
        calc_tools = [t for t in tools if t["function"]["name"] == "calculator"]
        result = ask_with_tools(query, tools=calc_tools)
        if result["tool_calls"]:
            for tc in result["tool_calls"]:
                trace.add_step("tool_call", {"tool": tc["tool"], "arguments": tc["arguments"]},
                               {"result": tc["result"]})
        trace.add_step("llm_call", {"query": query, "method": "tool_use"},
                       {"response": result["response"][:200]},
                       metadata={"tokens": result["total_tokens"]})
        answer = result["response"]
        total_tokens = result["total_tokens"]

    elif route == "internet_search":
        search_tools = [t for t in tools if t["function"]["name"] == "internet_search"]
        result = ask_with_tools(query, tools=search_tools)
        if result["tool_calls"]:
            for tc in result["tool_calls"]:
                trace.add_step("tool_call", {"tool": tc["tool"], "arguments": tc["arguments"]},
                               {"result": str(tc["result"])[:300]})
        trace.add_step("llm_call", {"query": query, "method": "tool_use"},
                       {"response": result["response"][:200]},
                       metadata={"tokens": result["total_tokens"]})
        answer = result["response"]
        total_tokens = result["total_tokens"]
    else:
        result = ask_direct(query)
        trace.add_step("llm_call", {"query": query, "method": "fallback"},
                       {"response": result["response"][:200]})
        answer = result["response"]
        total_tokens = result["total_tokens"]

    trace.final_answer = answer

    result_dict = {
        "query": query,
        "route": route,
        "answer": answer,
        "total_tokens": total_tokens
    }

    return result_dict, trace

In [24]:
# Run traced queries and save traces
traced_queries = [
    "What is chain-of-thought prompting?",
    "What is sqrt(98765)?",
    "Who won the most recent Super Bowl?",
    "What is the capital of France?",
]

all_traces = []
for q in traced_queries:
    result, trace = agent_answer_traced(q, faiss_index, indexed_chunks, embed_model)
    all_traces.append(trace)
    trace.save(f"trace_{len(all_traces)}.json")
    print(f"\nQ: {result['query']}")
    print(f"Route: {result['route']}")
    print(f"A: {result['answer'][:150]}")
    print(f"Steps: {len(trace.steps)}")
    print("-" * 50)

Trace saved to traces/trace_1.json

Q: What is chain-of-thought prompting?
Route: retrieval
A: Chain-of-thought (CoT) prompting is a technique that improves the reasoning ability of large language models by providing step-by-step reasoning examp
Steps: 3
--------------------------------------------------


Trace saved to traces/trace_2.json

Q: What is sqrt(98765)?
Route: calculator
A: The square root of 98765 is approximately 314.27.
Steps: 3
--------------------------------------------------


/var/folders/2x/lt99d9v97mgfkbc813jw5cdw0000gn/T/ipykernel_8829/2586427243.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Trace saved to traces/trace_3.json

Q: Who won the most recent Super Bowl?
Route: internet_search
A: The Kansas City Chiefs won the most recent Super Bowl, which was Super Bowl LVIII in 2024. They defeated the San Francisco 49ers.

Steps: 3
--------------------------------------------------


Trace saved to traces/trace_4.json

Q: What is the capital of France?
Route: direct
A: The capital of France is Paris.

Steps: 2
--------------------------------------------------


---

## Section 7: Trace Visualization

Load a saved trace from JSON and display it in a readable format.

In [25]:
def display_trace(trace_file):
    """Load and display a trace from a JSON file.

    Args:
        trace_file: path to the trace JSON file
    """
    with open(trace_file, "r") as f:
        trace_data = json.load(f)

    print("=" * 60)
    print(f"TRACE: {trace_data['query']}")
    print(f"Route: {trace_data.get('route', 'N/A')}")
    print(f"Total time: {trace_data.get('total_time', 0):.2f}s")
    print("=" * 60)

    for i, step in enumerate(trace_data["steps"]):
        print(f"\n  Step {i+1}: [{step['step_type'].upper()}]")
        print(f"    Time: {step.get('timestamp', 0):.2f}s")

        # Display input
        if isinstance(step["input"], dict):
            for k, v in step["input"].items():
                print(f"    Input {k}: {str(v)[:100]}")
        else:
            print(f"    Input: {str(step['input'])[:100]}")

        # Display output
        if isinstance(step["output"], dict):
            for k, v in step["output"].items():
                print(f"    Output {k}: {str(v)[:100]}")
        else:
            print(f"    Output: {str(step['output'])[:100]}")

        # Display metadata
        if "metadata" in step and step["metadata"]:
            for k, v in step["metadata"].items():
                print(f"    [{k}: {v}]")

    print(f"\nFinal Answer: {str(trace_data.get('final_answer', 'N/A'))[:200]}")
    print("=" * 60)

In [26]:
# Display all saved traces
trace_dir = Path("traces")
for i in range(1, len(all_traces) + 1):
    display_trace(trace_dir / f"trace_{i}.json")
    print()

TRACE: What is chain-of-thought prompting?
Route: retrieval
Total time: 1.35s

  Step 1: [ROUTE]
    Time: 0.46s
    Input query: What is chain-of-thought prompting?
    Output route: retrieval

  Step 2: [RETRIEVAL]
    Time: 0.54s
    Input query: What is chain-of-thought prompting?
    Output num_chunks: 3
    Output sources: ['chain_of_thought.txt', 'chain_of_thought.txt', 'prompt_engineering.txt']

  Step 3: [LLM_CALL]
    Time: 1.35s
    Input query: What is chain-of-thought prompting?
    Input method: rag
    Output response: Chain-of-thought (CoT) prompting is a technique that improves the reasoning ability of large languag
    [tokens: 453]

Final Answer: Chain-of-thought (CoT) prompting is a technique that improves the reasoning ability of large language models by providing step-by-step reasoning examples in the prompt. It encourages the model to brea

TRACE: What is sqrt(98765)?
Route: calculator
Total time: 1.22s

  Step 1: [ROUTE]
    Time: 0.28s
    Input query: What is 

---

## Section 8: Tool Description Experiments

How do tool descriptions affect the model's tool selection? We test with **minimal** and **expanded** descriptions.

### Define Alternative Tool Descriptions

In [27]:
TOOLS_MINIMAL = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate math expressions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "internet_search",
            "description": "Search the internet.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }
    }
]

TOOLS_EXPANDED = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Use this tool for any arithmetic calculations such as addition, subtraction, multiplication, division, exponentiation, square roots, or algebraic expressions. Examples: '15 * 37', '144 ** 0.5', '(3 + 4) * 2'. Always prefer this tool over mental math for accuracy.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The mathematical expression to evaluate. Use Python syntax: ** for exponents, * for multiplication, / for division."}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "internet_search",
            "description": "Use this tool to search the internet for factual information, recent events, current news, up-to-date statistics, or any information that may have changed after the model's training cutoff. Ideal for questions about recent developments, current leaders, latest scores, or real-time data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "A clear, specific search query that will return relevant results"}
                },
                "required": ["query"]
            }
        }
    }
]

print("Defined TOOLS_MINIMAL and TOOLS_EXPANDED")

Defined TOOLS_MINIMAL and TOOLS_EXPANDED


In [28]:
# Compare tool usage across description styles
description_test_queries = [
    "What is 2 + 2?",
    "What is the square root of 144?",
    "Who is the current president of the United States?",
    "What is 987 * 654?",
]

print("=" * 80)
print(f"{'Query':<45} {'Default':>10} {'Minimal':>10} {'Expanded':>10}")
print("=" * 80)

for q in description_test_queries:
    results = {}
    for label, tool_set in [("Default", TOOLS), ("Minimal", TOOLS_MINIMAL), ("Expanded", TOOLS_EXPANDED)]:
        r = ask_with_tools(q, tools=tool_set)
        tool_used = r["tool_calls"][0]["tool"] if r["tool_calls"] else "none"
        results[label] = tool_used
    
    print(f"{q:<45} {results['Default']:>10} {results['Minimal']:>10} {results['Expanded']:>10}")

print("=" * 80)

Query                                            Default    Minimal   Expanded


What is 2 + 2?                                calculator calculator calculator


What is the square root of 144?               calculator calculator calculator


/var/folders/2x/lt99d9v97mgfkbc813jw5cdw0000gn/T/ipykernel_8829/2586427243.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/var/folders/2x/lt99d9v97mgfkbc813jw5cdw0000gn/T/ipykernel_8829/2586427243.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/var/folders/2x/lt99d9v97mgfkbc813jw5cdw0000gn/T/ipykernel_8829/2586427243.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Who is the current president of the United States? internet_search internet_search internet_search


What is 987 * 654?                            calculator calculator calculator


**Observation:** Tool descriptions significantly influence whether and which tools the model chooses to call. More detailed descriptions tend to lead to more consistent tool usage, while minimal descriptions may cause the model to skip tool calls for simpler queries.

---

## Section 9: Evaluating Agent Decisions

We evaluate the agent's routing accuracy by comparing its chosen route against expected routes for a set of test queries.

In [29]:
EVAL_QUERIES = [
    {"query": "What is chain-of-thought prompting?", "expected_route": "retrieval"},
    {"query": "What is 256 * 384?", "expected_route": "calculator"},
    {"query": "Who won the 2024 Nobel Prize in Physics?", "expected_route": "internet_search"},
    {"query": "What is the capital of Japan?", "expected_route": "direct"},
    {"query": "What is retrieval augmented generation?", "expected_route": "retrieval"},
    {"query": "What is sqrt(12345)?", "expected_route": "calculator"},
    {"query": "What are the latest developments in AI regulation?", "expected_route": "internet_search"},
    {"query": "Explain how photosynthesis works.", "expected_route": "direct"},
]

In [30]:
# Run evaluation
print("=" * 90)
print(f"{'Query':<50} {'Expected':>15} {'Actual':>15} {'Correct':>8}")
print("=" * 90)

correct = 0
eval_results = []

for item in EVAL_QUERIES:
    query = item["query"]
    expected = item["expected_route"]
    
    result = agent_answer(query, faiss_index, indexed_chunks, embed_model)
    actual = result["route"]
    is_correct = actual == expected
    if is_correct:
        correct += 1
    
    eval_results.append({
        "query": query,
        "expected": expected,
        "actual": actual,
        "correct": is_correct,
        "answer": result["answer"][:100]
    })
    
    mark = "Y" if is_correct else "N"
    print(f"{query:<50} {expected:>15} {actual:>15} {mark:>8}")

accuracy = correct / len(EVAL_QUERIES)
print("=" * 90)
print(f"\nRoute Accuracy: {correct}/{len(EVAL_QUERIES)} = {accuracy:.1%}")

Query                                                     Expected          Actual  Correct


What is chain-of-thought prompting?                      retrieval       retrieval        Y


What is 256 * 384?                                      calculator      calculator        Y


/var/folders/2x/lt99d9v97mgfkbc813jw5cdw0000gn/T/ipykernel_8829/2586427243.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Who won the 2024 Nobel Prize in Physics?           internet_search internet_search        Y


...

What is retrieval augmented generation?                  retrieval       retrieval        Y


What is sqrt(12345)?                                    calculator      calculator        Y


/var/folders/2x/lt99d9v97mgfkbc813jw5cdw0000gn/T/ipykernel_8829/2586427243.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


What are the latest developments in AI regulation? internet_search internet_search        Y


Explain how photosynthesis works.                           direct          direct        Y

Route Accuracy: 8/8 = 100.0%


---

## Reflection Questions

Think about the following questions based on your experiments:

1. **When did retrieval improve answers?** RAG improved answers for questions about specific topics covered in the document corpus (e.g., chain-of-thought prompting, RAG itself). It provided more precise, grounded answers compared to direct QA.

2. **When did tool calls help?** Tool calls were essential for math computations (the model cannot reliably compute square roots of large numbers) and for current events (the model's training data has a cutoff).

3. **When did the agent choose the wrong route?** The router sometimes misclassified ambiguous queries, such as questions that could be answered both from the corpus and from general knowledge.

4. **How did tool descriptions affect tool selection?** More detailed descriptions led to more consistent and appropriate tool usage. Minimal descriptions sometimes caused the model to attempt answering without tools even when a tool would have been more accurate.

5. **What information in the trace helped debug failures?** The trace reveals the routing decision, which tool was called (if any), what arguments were passed, and the tool's output. This makes it easy to identify whether a failure occurred at the routing stage, the tool execution stage, or the final generation stage.

---

## Congratulations!

You have built a complete agent system that:

1. **Answers directly** from parametric knowledge
2. **Retrieves documents** using RAG with embeddings and vector search
3. **Calls tools** including a calculator and internet search
4. **Routes queries** to the appropriate handler
5. **Logs traces** for debugging and analysis
6. **Visualizes execution** step by step
7. **Experiments with tool descriptions** to understand their impact
8. **Evaluates routing accuracy** systematically

The key takeaway: **modern AI systems are not just language models. They are orchestration systems that combine models with retrieval, tools, and structured interfaces to interact with the world.**